# Bombyx mori piRNA smRNA-seq analysis

This notebook mainly records the smRNA-seq analysis methods for samples after knockout.

In [31]:
from pathlib import Path

from bm_pirna.config import RAW_DATA_DIR,INTERIM_DATA_DIR,REPORTS_DIR,EXTERNAL_DATA_DIR,PROCESSED_DATA_DIR
from bm_pirna.utils import run_cmd

## Notebook Configuration

In [34]:
task_name = "smRNA-seq_20260122"

#qc
sample_map = RAW_DATA_DIR / "sample_map.csv"
sample_dir = RAW_DATA_DIR / "pirna/bm_smRNA_clean_data"
rename_output = INTERIM_DATA_DIR / task_name / "renamed"
smrna_seq_adapter = "AACTGTAGGCACCATCAAT"
qc_output = INTERIM_DATA_DIR / task_name / "qc"
qc_report_dir = REPORTS_DIR / task_name / "fastp"

#filter
structure_rna = EXTERNAL_DATA_DIR / "extracted_gene_sequences/bm_NCBI_fasta/filtered_non_piRNA.miRNA_rRNA_snRNA_snoRNA_tRNA.fasta"
structure_rna_index = str(INTERIM_DATA_DIR / task_name / "bowtie_index/structure_rna_index")
structure_rna_filtered_output = INTERIM_DATA_DIR / task_name / "structure_rna_filtered"

#Conventional piRNA determined
transposon_fasta = EXTERNAL_DATA_DIR / "TE-Silkworm.fa"
transposon_index = str(INTERIM_DATA_DIR / task_name / "bowtie_index/transposon_index")
transposon_filtered_output = INTERIM_DATA_DIR / task_name / "transposon_filtered"

#stats
transposon_dedrived_pirna_stats_output = PROCESSED_DATA_DIR / task_name / "trasposon_dedrived_pirna_stats"
clean_reads_stats_output = PROCESSED_DATA_DIR / task_name / "clean_reads_stats"


## Quality control
Remove low-quality reads and adapter sequences from raw sequencing data.

In [6]:
# sample rename
rename_cmd = f"pixi run python -m bm_pirna.smrna_seq.rename {sample_map} -i {sample_dir} -o {rename_output}"
run_cmd(
    rename_cmd,
    [rename_output]
)


2026-01-22 22:05:56.602 | INFO     | bm_pirna.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/bm_pirna


2026-01-22 22:05:56.753 | INFO     | __main__:main:56 - Control_1.fq.gz -> WT_rep1.fq.gz
2026-01-22 22:05:56.868 | INFO     | __main__:main:56 - RNPS1-KD1_1.fq.gz -> RNPS1_rep1.fq.gz
2026-01-22 22:05:56.869 | INFO     | __main__:main:56 - SUGP1-KD2_1.fq.gz -> SUGP1_rep1.fq.gz
2026-01-22 22:05:56.869 | INFO     | __main__:main:56 - DHX15-KD3_1.fq.gz -> DHX15_rep1.fq.gz
2026-01-22 22:05:56.869 | SUCCESS  | __main__:main:58 - Symlinks created in /home/gyk/project/bm_pirna/data/interim/smRNA-seq_20260122/renamed


In [36]:
# qc
qc_cmd = f"pixi run python -m bm_pirna.smrna_seq.qc --force -o {qc_output} -a {smrna_seq_adapter} -l 24-35 --quality 20 --overlap 5 --keep-untrimmed --cores 16 --pattern *.f*q.gz --report-dir {qc_report_dir} {rename_output}"
run_cmd(
    qc_cmd,
    [qc_output, qc_report_dir],
    force=True
)

2026-01-23 22:53:08.288 | INFO     | bm_pirna.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/bm_pirna


2026-01-23 22:53:08.442 | INFO     | __main__:main:366 - Length filter enabled: 24-35bp
2026-01-23 22:53:08.444 | INFO     | __main__:main:369 - Filtered output dir: /home/gyk/project/bm_pirna/data/interim/smRNA-seq_20260122/qc/24-35_length_filtered
2026-01-23 22:53:08.444 | INFO     | __main__:main:387 - Processing 4 file(s) with adapter: AACTGTAGGCACCATCAAT
2026-01-23 22:53:08.445 | INFO     | __main__:qc_single_file:235 - [cutadapt] Trimming adapter from DHX15_rep1.fq.gz
2026-01-23 22:53:08.445 | DEBUG    | __main__:run_cutadapt:44 - Running: cutadapt -a AACTGTAGGCACCATCAAT --overlap 5 -j 16 -o /tmp/tmpkp9lsstp/DHX15_rep1.trim.fq.gz /home/gyk/project/bm_pirna/data/interim/smRNA-seq_20260122/renamed/DHX15_rep1.fq.gz
2026-01-23 22:53:22.913 | INFO     | __main__:qc_single_file:246 - [fastp] Quality filtering DHX15_rep1
2026-01-23 22:53:22.913 | DEBUG    | __main__:run_fastp:83 - Running: fastp -i /tmp/tmpkp9lsstp/DHX15_rep1.trim.fq.gz -o /home/gyk/project/bm_pirna/data/interim/smRNA-s

## Structural RNA filtering
Filtering out structural RNAs, such as tRNA, rRNA, snRNA, and snoRNA. The sequences of the structural RNAs of Bombyx mori is extracted from the [NCBI genome annotation](https://www.ncbi.nlm.nih.gov/refseq/annotation_euk/Bombyx_mori/GCF_030269925.1-RS_2024_01/).

In [22]:
structure_rna_index_cmd = f"pixi run bowtie-build -f --quiet {structure_rna} {structure_rna_index}"
structure_rna_fiter_cmd = f"pixi run python -m bm_pirna.smrna_seq.filter {qc_output}/24-35_length_filtered {structure_rna_index} -v 1 -m exclude --threads 16 --output-dir {structure_rna_filtered_output} --suffix .fq.gz"

run_cmd(structure_rna_index_cmd,[Path(f"{structure_rna_index}.1.ebwt")])
run_cmd(structure_rna_fiter_cmd,[structure_rna_filtered_output])

⏭️ /home/gyk/project/bm_pirna/data/interim/smRNA-seq_20260122/bowtie_index/structure_rna_index.1.ebwt exists, skip


2026-01-23 17:21:07.850 | INFO     | bm_pirna.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/bm_pirna


2026-01-23 17:21:08.006 | INFO     | __main__:main:248 - Filtering 4 file(s) using index: structure_rna_index
2026-01-23 17:21:08.007 | INFO     | __main__:main:249 - Mode: exclude
2026-01-23 17:21:08.008 | INFO     | __main__:filter_single_file:107 - [bowtie] Filtering DHX15_rep1.filtered.fq.gz (exclude mode)
2026-01-23 17:21:08.008 | DEBUG    | __main__:run_bowtie:64 - Running: bowtie -p 16 -v 1 -q -S --best --un /tmp/tmpgt6tzpuo/bowtie_output.fq /home/gyk/project/bm_pirna/data/interim/smRNA-seq_20260122/bowtie_index/structure_rna_index /home/gyk/project/bm_pirna/data/interim/smRNA-seq_20260122/qc/24-35_length_filtered/DHX15_rep1.filtered.fq.gz
2026-01-23 17:21:30.492 | INFO     | __main__:filter_single_file:135 - [pigz] Compressing to DHX15_rep1.fq.gz
2026-01-23 17:21:30.493 | DEBUG    | __main__:compress_with_pigz:84 - Running: pigz -c /tmp/tmpgt6tzpuo/bowtie_output.fq > /home/gyk/project/bm_pirna/data/interim/smRNA-seq_20260122/structure_rna_filtered/DHX15_rep1.fq.gz
2026-01-23 17

## Conventional piRNA determined

Retain only sequences that can be aligned to the transposon, with no mismatches allowed.

In [ ]:
transposon_index_cmd = f"pixi run bowtie-build -f --quiet {transposon_fasta} {transposon_index}"
transposon_filter_cmd = f"pixi run python -m bm_pirna.smrna_seq.filter  {structure_rna_filtered_output} {transposon_index} -v 0 -m include --threads 16 --output-dir {transposon_filtered_output} --suffix .transposon.fq.gz"

run_cmd(transposon_index_cmd,[Path(f"{transposon_index}.1.ebwt")])
run_cmd(transposon_filter_cmd,[transposon_filtered_output])

2026-01-23 19:49:43.934 | INFO     | bm_pirna.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/bm_pirna


2026-01-23 19:49:44.095 | INFO     | __main__:main:248 - Filtering 4 file(s) using index: transposon_index
2026-01-23 19:49:44.097 | INFO     | __main__:main:249 - Mode: include
2026-01-23 19:49:44.097 | INFO     | __main__:filter_single_file:107 - [bowtie] Filtering DHX15_rep1.fq.gz (include mode)
2026-01-23 19:49:44.097 | DEBUG    | __main__:run_bowtie:64 - Running: bowtie -p 16 -v 0 -q -S --best --al /tmp/tmp34_1k0ei/bowtie_output.fq /home/gyk/project/bm_pirna/data/interim/smRNA-seq_20260122/bowtie_index/transposon_index /home/gyk/project/bm_pirna/data/interim/smRNA-seq_20260122/structure_rna_filtered/DHX15_rep1.fq.gz
2026-01-23 19:49:55.360 | INFO     | __main__:filter_single_file:135 - [pigz] Compressing to DHX15_rep1.transposon.fq.gz
2026-01-23 19:49:55.362 | DEBUG    | __main__:compress_with_pigz:84 - Running: pigz -c /tmp/tmp34_1k0ei/bowtie_output.fq > /home/gyk/project/bm_pirna/data/interim/smRNA-seq_20260122/transposon_filtered/DHX15_rep1.transposon.fq.gz
2026-01-23 19:49:55.

## Reads stats
Statistics of the filtered sequence information

In [35]:
transposon_stats_cmd = f"python -m bm_pirna.smrna_seq.stats {transposon_filtered_output} -o {transposon_dedrived_pirna_stats_output}"
clean_reads_stats_cmd = f"python -m bm_pirna.smrna_seq.stats {structure_rna_filtered_output} -o {clean_reads_stats_output}"

run_cmd(transposon_stats_cmd, [transposon_dedrived_pirna_stats_output])
run_cmd(clean_reads_stats_cmd, [clean_reads_stats_output])

⏭️ /home/gyk/project/bm_pirna/data/processed/smRNA-seq_20260122/trasposon_dedrived_pirna_stats exists, skip


2026-01-23 21:58:14.037 | INFO     | bm_pirna.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/bm_pirna


2026-01-23 21:58:14.366 | INFO     | __main__:main:263 - Analyzing 4 file(s)
2026-01-23 21:58:14.368 | INFO     | __main__:main:268 - [1/4] Analyzing DHX15_rep1.fq.gz
2026-01-23 22:00:50.178 | DEBUG    | __main__:main:271 -   Reads: 7,764,509, Length: 24-35, GC: 43.1%
2026-01-23 22:00:50.179 | INFO     | __main__:main:268 - [2/4] Analyzing RNPS1_rep1.fq.gz
2026-01-23 22:03:46.599 | DEBUG    | __main__:main:271 -   Reads: 8,774,779, Length: 24-35, GC: 43.4%
2026-01-23 22:03:46.600 | INFO     | __main__:main:268 - [3/4] Analyzing SUGP1_rep1.fq.gz
2026-01-23 22:06:39.884 | DEBUG    | __main__:main:271 -   Reads: 8,520,145, Length: 24-35, GC: 43.1%
2026-01-23 22:06:39.884 | INFO     | __main__:main:268 - [4/4] Analyzing WT_rep1.fq.gz
2026-01-23 22:10:04.512 | DEBUG    | __main__:main:271 -   Reads: 10,352,523, Length: 24-35, GC: 40.7%
2026-01-23 22:10:04.512 | INFO     | __main__:main:278 - Writing output files...
2026-01-23 22:10:09.041 | SUCCESS  | __main__:main:293 - All stats saved to:

## Genome mapping

Align the filtered reads to the [silkworm genome](https://silkbase.ab.a.u-tokyo.ac.jp/cgi-bin/index.cgi).